# 09 — Filter BCN20000 IDs from Mel+Nevus Histo

**Issue:** D4.2 (#138)  
**Depends on:** D4.1 (#137)  
**Purpose:** Remove BCN20000 duplicate images from MNH by exact isic_id match before merge.

In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path("/Users/rehmaaziz/revela")

BCN_TRAIN = ROOT / "data/processed/bcn20000_cancer_risk/train.csv"
BCN_VAL   = ROOT / "data/processed/bcn20000_cancer_risk/val.csv"
BCN_TEST  = ROOT / "data/processed/bcn20000_cancer_risk/test.csv"
MNH_META  = ROOT / "data/mel_nevus_histo/metadata.csv"
MNH_OUT   = ROOT / "data/mel_nevus_histo/mnh_filtered.csv"

bcn_splits = pd.concat([
    pd.read_csv(BCN_TRAIN),
    pd.read_csv(BCN_VAL),
    pd.read_csv(BCN_TEST),
], ignore_index=True)

bcn_ids = set(bcn_splits["isic_id"])
print(f"BCN20000 isic_ids loaded: {len(bcn_ids):,}")

BCN20000 isic_ids loaded: 17,639


In [2]:
mnh = pd.read_csv(MNH_META)
print(f"MNH before filter: {len(mnh):,} rows")

mnh_filtered = mnh[~mnh["isic_id"].isin(bcn_ids)].reset_index(drop=True)
removed = len(mnh) - len(mnh_filtered)
print(f"Removed (BCN overlap): {removed:,}")
print(f"MNH after filter: {len(mnh_filtered):,} rows")

MNH before filter: 18,133 rows
Removed (BCN overlap): 5,633
MNH after filter: 12,500 rows


/var/folders/9_/n4_t7jmn2dx95gjqy9bjxbmh0000gn/T/ipykernel_28241/300135245.py:1: DtypeWarning: Columns (0: anatom_site_4, 1: anatom_site_5) have mixed types. Specify dtype option on import or set low_memory=False.
  mnh = pd.read_csv(MNH_META)


In [3]:
assert len(set(mnh_filtered["isic_id"]) & bcn_ids) == 0, "BCN overlap remains — FAIL"
print("Assert passed: zero BCN overlap confirmed")

Assert passed: zero BCN overlap confirmed


In [4]:
def mel_count(df):
    mask = df["diagnosis_2"].str.contains("melanoma", case=False, na=False) | \
           df["diagnosis_3"].str.contains("melanoma", case=False, na=False)
    return mask.sum()

print(f"Melanoma before filter: {mel_count(mnh):,}")
print(f"Melanoma after filter:  {mel_count(mnh_filtered):,}")

Melanoma before filter: 7,191
Melanoma after filter:  4,334


In [5]:
mnh_filtered.to_csv(MNH_OUT, index=False)
print(f"Saved: {MNH_OUT}")

Saved: /Users/rehmaaziz/revela/data/mel_nevus_histo/mnh_filtered.csv
